###Практична робота. Delta-таблиця: зміни даних, історія версій і time travel###

#### Завдання 1. Довідник клієнтів у форматі Delta ####

In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.customers_delta;


In [0]:
%sql
CREATE TABLE workspace.default.customers_delta 
AS SELECT 
    c_custkey, 
    c_name, 
    c_nationkey, 
    c_mktsegment, 
    c_acctbal
FROM samples.tpch.customer
WHERE c_custkey BETWEEN 1 AND 20;


num_affected_rows,num_inserted_rows


In [0]:
%sql
DESCRIBE DETAIL workspace.default.customers_delta;


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,29238f75-1886-435b-81ac-70a1d69775a3,workspace.default.customers_delta,null,,2026-10-29T18:03:09.524Z,2026-10-29T18:03:12.000Z,List(),List(),1,2178,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


#### Завдання 2. Зміни в довіднику ####


In [0]:
%sql
INSERT INTO workspace.default.customers_delta
SELECT 
    c_custkey, 
    c_name, 
    c_nationkey, 
    c_mktsegment, 
    c_acctbal
FROM samples.tpch.customer
WHERE c_custkey BETWEEN 21 AND 25;


num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
UPDATE workspace.default.customers_delta
SET c_mktsegment = 'AUTOMOBILE'
WHERE c_custkey = 1;


num_affected_rows
1


In [0]:
%sql
DELETE FROM workspace.default.customers_delta
WHERE c_acctbal < 0;


num_affected_rows
1


In [0]:
%sql
SELECT * 
FROM workspace.default.customers_delta
ORDER BY c_custkey ASC;


c_custkey,c_name,c_nationkey,c_mktsegment,c_acctbal
1,Customer#000000001,15,AUTOMOBILE,711.56
2,Customer#000000002,13,AUTOMOBILE,121.65
3,Customer#000000003,1,AUTOMOBILE,7498.12
4,Customer#000000004,4,MACHINERY,2866.83
5,Customer#000000005,3,HOUSEHOLD,794.47
6,Customer#000000006,20,AUTOMOBILE,7638.57
7,Customer#000000007,18,AUTOMOBILE,9561.95
8,Customer#000000008,17,BUILDING,6819.74
9,Customer#000000009,8,FURNITURE,8324.07
10,Customer#000000010,5,HOUSEHOLD,2753.54


#### Завдання 3. Пакет змін із CRM####


In [0]:
%sql
MERGE INTO workspace.default.customers_delta AS t
USING (
    SELECT 
        c_custkey, 
        c_name, 
        c_nationkey, 
        c_mktsegment, 
        (c_acctbal + 100) AS c_acctbal
    FROM samples.tpch.customer
    WHERE c_custkey BETWEEN 21 AND 30
) AS s
ON t.c_custkey = s.c_custkey
WHEN MATCHED THEN
    UPDATE SET *
WHEN NOT MATCHED THEN
    INSERT *;


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
10,5,0,5


In [0]:
%sql
SELECT * 
FROM workspace.default.customers_delta
ORDER BY c_custkey ASC;


c_custkey,c_name,c_nationkey,c_mktsegment,c_acctbal
1,Customer#000000001,15,AUTOMOBILE,711.56
2,Customer#000000002,13,AUTOMOBILE,121.65
3,Customer#000000003,1,AUTOMOBILE,7498.12
4,Customer#000000004,4,MACHINERY,2866.83
5,Customer#000000005,3,HOUSEHOLD,794.47
6,Customer#000000006,20,AUTOMOBILE,7638.57
7,Customer#000000007,18,AUTOMOBILE,9561.95
8,Customer#000000008,17,BUILDING,6819.74
9,Customer#000000009,8,FURNITURE,8324.07
10,Customer#000000010,5,HOUSEHOLD,2753.54


In [0]:
# 1. Получаем полную историю изменений Delta-таблицы
history_df = spark.sql("DESCRIBE HISTORY workspace.default.customers_delta")

# 2. Выбираем только требуемые аудитором колонки
auditor_history_df = history_df.select("version", "operation", "operationMetrics")

# 3. Сортируем строки по номеру версии по возрастанию
sorted_history_df = auditor_history_df.sort("version")

# 4. Выводим результат в виде интерактивной таблицы
display(sorted_history_df)


version,operation,operationMetrics
0,CREATE TABLE AS SELECT,"Map(numFiles -> 1, numOutputRows -> 20, numOutputBytes -> 2178)"
1,WRITE,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1978)"
2,UPDATE,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3182, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1659, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1859, rewriteTimeMs -> 1509)"
3,DELETE,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1031, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 605, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 425)"
4,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2100, numTargetBytesRemoved -> 1978, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 2629, materializeSourceTimeMs -> 7, numTargetRowsInserted -> 5, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1205, numTargetRowsUpdated -> 5, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 10, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1358)"


#### Завдання 5. Time travel ####


#### Завдання 5.1. Клієнти і баланс за версіями ####

In [0]:
%sql
SELECT 0 AS version, COUNT(*) AS customers_cnt, SUM(c_acctbal) AS total_acctbal FROM workspace.default.customers_delta VERSION AS OF 0
UNION ALL
SELECT 1 AS version, COUNT(*) AS customers_cnt, SUM(c_acctbal) AS total_acctbal FROM workspace.default.customers_delta VERSION AS OF 1
UNION ALL
SELECT 2 AS version, COUNT(*) AS customers_cnt, SUM(c_acctbal) AS total_acctbal FROM workspace.default.customers_delta VERSION AS OF 2
UNION ALL
SELECT 3 AS version, COUNT(*) AS customers_cnt, SUM(c_acctbal) AS total_acctbal FROM workspace.default.customers_delta VERSION AS OF 3
UNION ALL
SELECT 4 AS version, COUNT(*) AS customers_cnt, SUM(c_acctbal) AS total_acctbal FROM workspace.default.customers_delta VERSION AS OF 4
ORDER BY version ASC;


version,customers_cnt,total_acctbal
0,20,88826.46
1,25,110568.08
2,25,110568.08
3,24,110840.68
4,29,140649.03


#### Завдання 5.2. Стан на момент видалення ####


In [0]:
# 1. Отримуємо історію таблиці та знаходимо рядок з операцією DELETE
history_df = spark.sql("DESCRIBE HISTORY workspace.default.customers_delta")
delete_row = history_df.filter("operation = 'DELETE'").select("timestamp").collect()

# 2. Зберігаємо значення timestamp у змінну Python (форматуємо як рядок)
delete_timestamp = str(delete_row[0]['timestamp'])

# 3. Виконуємо динамічний SQL-запит за допомогою f-рядка та конструкції TIMESTAMP AS OF
result_df = spark.sql(f"""
    SELECT 
        CAST('{delete_timestamp}' AS TIMESTAMP) AS as_of, 
        COUNT(*) AS customers_cnt 
    FROM workspace.default.customers_delta TIMESTAMP AS OF '{delete_timestamp}'
""")

# 4. Виводимо результат
display(result_df)


as_of,customers_cnt
2026-10-29T18:03:23.000Z,24


#### Завдання 6. Помилка і відновлення ####

#### Завдання 6.1. Повернення до попередньої версії ####

In [0]:
%sql
-- 1. Симулюємо помилку оператора CRM (створюється версія 5)
UPDATE workspace.default.customers_delta
SET c_acctbal = 0;

-- 2. Відновлюємо таблицю до версії 4 (створюється версія 6)
RESTORE TABLE workspace.default.customers_delta TO VERSION AS OF 4;


table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
6137,3,1,3,2191,6137


#### Завдання 6.2. Перевірка повернення ####

In [0]:
%sql
SELECT 
    (
        SELECT COUNT(*) FROM (
            SELECT * FROM workspace.default.customers_delta
            EXCEPT
            SELECT * FROM workspace.default.customers_delta VERSION AS OF 4
        )
    ) AS only_current,
    (
        SELECT COUNT(*) FROM (
            SELECT * FROM workspace.default.customers_delta VERSION AS OF 4
            EXCEPT
            SELECT * FROM workspace.default.customers_delta
        )
    ) AS only_merge_version;


only_current,only_merge_version
0,0


#### Завдання 7. Обслуговування таблиці ####


In [0]:
%sql
DESCRIBE DETAIL workspace.default.customers_delta;


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,29238f75-1886-435b-81ac-70a1d69775a3,workspace.default.customers_delta,null,,2026-10-29T18:03:09.524Z,2026-10-29T18:03:54.000Z,List(),List(),3,6137,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 2, numDeletionVectors -> 1)",false


In [0]:
%sql
OPTIMIZE workspace.default.customers_delta;


path,metrics
,"List(1, 3, List(2255, 2255, 2255.0, 1, 2255), List(1859, 2178, 2045.6666666666667, 3, 6137), 0, null, null, 0, 1, 3, 0, true, 0, 0, 1793297041863, 1793297044734, 8, 1, null, List(1, 2), null, 5, 5, 361, 0, null, null, 0)"


In [0]:
%sql
DESCRIBE DETAIL workspace.default.customers_delta;


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,29238f75-1886-435b-81ac-70a1d69775a3,workspace.default.customers_delta,null,,2026-10-29T18:03:09.524Z,2026-10-29T18:04:05.000Z,List(),List(),1,2255,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
VACUUM workspace.default.customers_delta;


path
""


In [0]:
# Отримуємо оновлену історію змін Delta-таблиці
final_history_df = spark.sql("DESCRIBE HISTORY workspace.default.customers_delta")

# Фільтруємо за вимогами аудитора та сортуємо за зростанням версії
display(
    final_history_df
    .select("version", "operation", "operationMetrics")
    .sort("version")
)


version,operation,operationMetrics
0,CREATE TABLE AS SELECT,"Map(numFiles -> 1, numOutputRows -> 20, numOutputBytes -> 2178)"
1,WRITE,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1978)"
2,UPDATE,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3182, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1659, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1859, rewriteTimeMs -> 1509)"
3,DELETE,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1031, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 605, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 425)"
4,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2100, numTargetBytesRemoved -> 1978, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 2629, materializeSourceTimeMs -> 7, numTargetRowsInserted -> 5, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1205, numTargetRowsUpdated -> 5, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 10, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1358)"
5,UPDATE,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6137, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 987, numDeletionVectorsUpdated -> 0, scanTimeMs -> 10, numAddedFiles -> 1, numUpdatedRows -> 29, numAddedBytes -> 2191, rewriteTimeMs -> 977)"
6,RESTORE,"Map(numRestoredFiles -> 3, removedFilesSize -> 2191, numRemovedFiles -> 1, restoredFilesSize -> 6137, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 3, tableSizeAfterRestore -> 6137)"
7,OPTIMIZE,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6137, p25FileSize -> 2255, numDeletionVectorsRemoved -> 1, minFileSize -> 2255, numAddedFiles -> 1, maxFileSize -> 2255, p75FileSize -> 2255, p50FileSize -> 2255, numAddedBytes -> 2255)"
8,VACUUM START,"Map(numFilesToDelete -> 0, sizeOfDataToDelete -> 0)"
9,VACUUM END,"Map(numDeletedFiles -> 0, numVacuumedDirectories -> 1)"
